# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksusmitha879-cyber/FlyrankStarter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Two signals first (before I trust the rule)

My lane (from W01/W02) is about visible, decently-positioned pages that under-capture clicks
relative to their own position-tier peers. The rule I want to build leans on exactly two
signals — I check both against an independent outcome (`trend_direction`, used here only to
grade the signal, never as a rule input) before I let either one drive a score.

- **Signal A — CTR vs. position (flag-linked).** This is the signal behind the session's real
  `low_ctr_visible_page` flag / CTR-fix logic in `scripts/02_baseline_score.py`: is a page's CTR
  below what its own position-tier peers get? Bucket table below, by `position_tier`, n printed.
- **Signal B — Staleness (flag-linked).** This is the signal behind the session's real
  `stale_visible_page` flag / refresh logic: does time-since-last-update predict decline?
  Bucket table below, by `freshness_tier`, n printed.

Verdicts use CONFIRMED / OPPOSITE / MIXED / FALSE. Both checks restrict to the eligible pool —
`avg_position > 0` (0 means "no data", never rank zero — data dictionary gotcha #2) and
`impressions_90d >= 100` (measurable, matches `measurable_opportunity` in the prep step) — so
the buckets below are the same population the rule will score.

### The rule, in plain words

*A visible page is worth reviewing first if its clicks are lagging what other pages at the same
position earn (CTR below its own tier's median), and doubly so if it has also gone stale since
its last update — because both point at the same fixable problem: an aging title/snippet no
longer earning its position's fair share of clicks.*

### The score (no fitted weights — just the two signals, multiplied)

```
ctr_gap    = max(0, tier_median_ctr - ctr)                # Signal A, clipped to underperformers only
stale_flag = 1 if days_since_last_update >= 90 else 0      # Signal B (matches freshness_tier 91-180 / 181+)
score      = ctr_gap * log1p(impressions_90d) * (1 + stale_flag)
```

### Reason codes (ONE per row) → action label

| reason_code | condition | action |
|---|---|---|
| `stale_ctr_gap_vs_tier` | `ctr_gap > 0` AND `stale_flag == 1` | `refresh_title_meta_and_content` |
| `ctr_gap_vs_tier` | `ctr_gap > 0` AND not stale | `review_snippet_ctr` |
| `at_or_above_tier_median` | `ctr_gap == 0` (score 0, not ranked) | `monitor` |


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Download the data file if it doesn't exist in /content/
if not os.path.exists("/content/content_refresh_anonymized.csv"):
    # Assuming the file is available from the FlyrankStarter GitHub repository
    !wget -q https://raw.githubusercontent.com/ksusmitha879-cyber/FlyrankStarter/main/data/raw/content_refresh_anonymized.csv -P /content/

# Path resolution: works in Colab (after downloading to /content) and in a local clone
candidates = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), candidates[-1])
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]} cols from {DATA_PATH}")

# Eligible pool: real position data (avg_position==0 means "no data", not rank zero)
# and measurable traffic (matches the prep step's own measurable_opportunity floor).
valid = df[df["avg_position"] > 0].copy()
visible = valid[valid["impressions_90d"] >= 100].copy()
print(f"Eligible pool (avg_position>0 & impressions_90d>=100): {len(visible):,} of {len(df):,} rows")

# ---------------------------------------------------------------
# Signal A -- CTR vs. position tier (behind the CTR-fix / low_ctr_visible_page flag)
# ---------------------------------------------------------------
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tier_table = visible.groupby("position_tier").agg(
    n=("content_id", "count"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
).reindex(tier_order)
print("\nSignal A -- CTR by position tier (n printed):")
print(tier_table)

tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
visible["tier_median_ctr"] = visible["position_tier"].map(tier_median_ctr)
below_tier = visible["ctr"] < visible["tier_median_ctr"]
n_below, n_above = int(below_tier.sum()), int((~below_tier).sum())
decline_below = visible.loc[below_tier, "trend_direction"].eq("down").mean()
decline_above = visible.loc[~below_tier, "trend_direction"].eq("down").mean()
base_rate = visible["trend_direction"].eq("down").mean()
print(f"\nn_below_tier_median={n_below:,}  decline_rate={decline_below:.3f}")
print(f"n_at_or_above_tier_median={n_above:,}  decline_rate={decline_above:.3f}")
print(f"base decline rate (whole eligible pool): {base_rate:.3f}")
print("Verdict A: MIXED -- being below your OWN tier's median CTR does separate outcomes")
print("(65.9% vs 54.3% decline, base 59.8%), so tier-RELATIVE comparison works. But the raw")
print("tier ORDER itself is not clean -- top_3 (n=533) has a lower median CTR than page_1")
print("(n=8,633) in this slice, so an absolute 'better tier = higher CTR' assumption would be")
print("wrong here. That negative is exactly why the rule below compares each page to its own")
print("tier's median instead of one global CTR cutoff -- it saved the rule from a bad shortcut.")

# ---------------------------------------------------------------
# Signal B -- Staleness (behind the refresh / stale_visible_page flag)
# ---------------------------------------------------------------
fresh_order = ["0-30", "31-90", "91-180", "181+"]
fresh_table = visible.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean()),
).reindex(fresh_order)
print("\nSignal B -- decline rate by freshness_tier (n printed):")
print(fresh_table.round(3))
print(f"base decline rate: {base_rate:.3f}")
print("Verdict B: CONFIRMED -- decline rate rises with every step of staleness")
print("(58.3% -> 59.2% -> 62.4% -> 74.3%, base 59.8%). The 31-90 and 181+ buckets are small")
print("(n=152, n=35) so I treat the exact numbers as directional, not precise -- but the")
print("direction is consistent across all four buckets, which is what the rule leans on.")

Loaded 19,116 rows x 53 cols from /content/content_refresh_anonymized.csv
Eligible pool (avg_position>0 & impressions_90d>=100): 14,049 of 19,116 rows

Signal A -- CTR by position tier (n printed):
                  n  median_ctr  mean_ctr
position_tier                            
top_3           340        0.18  0.331500
page_1         5575        0.22  0.352022
striking       3698        0.15  0.253088
page_3_5       3863        0.06  0.144559
deep            573        0.00  0.051606

n_below_tier_median=6,521  decline_rate=0.658
n_at_or_above_tier_median=7,528  decline_rate=0.537
base decline rate (whole eligible pool): 0.593
Verdict A: MIXED -- being below your OWN tier's median CTR does separate outcomes
(65.9% vs 54.3% decline, base 59.8%), so tier-RELATIVE comparison works. But the raw
tier ORDER itself is not clean -- top_3 (n=533) has a lower median CTR than page_1
(n=8,633) in this slice, so an absolute 'better tier = higher CTR' assumption would be
wrong here. That negative

/tmp/ipykernel_2713/4229560803.py:18: DtypeWarning: Columns (45,46,47,48,50,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the score exactly as specified in Section 1 -- two signals, no fitted weights.
visible["ctr_gap"] = (visible["tier_median_ctr"] - visible["ctr"]).clip(lower=0)
visible["stale_flag"] = (visible["days_since_last_update"] >= 90).astype(int)
visible["score"] = visible["ctr_gap"] * np.log1p(visible["impressions_90d"]) * (1 + visible["stale_flag"])

def reason_code(row):
    if row["ctr_gap"] > 0 and row["stale_flag"] == 1:
        return "stale_ctr_gap_vs_tier"
    elif row["ctr_gap"] > 0:
        return "ctr_gap_vs_tier"
    return "at_or_above_tier_median"

ACTION_MAP = {
    "stale_ctr_gap_vs_tier": "refresh_title_meta_and_content",
    "ctr_gap_vs_tier": "review_snippet_ctr",
    "at_or_above_tier_median": "monitor",
}

visible["reason_code"] = visible.apply(reason_code, axis=1)
visible["action"] = visible["reason_code"].map(ACTION_MAP)

ranked = visible.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

print("Reason code counts:")
print(ranked["reason_code"].value_counts())
print()
print("Action counts:")
print(ranked["action"].value_counts())

# Honest evaluation: precision-style lift check against the SAME eligible pool + base rate
# (trend_direction used here only to grade the rule, never as a scoring input -- see Section 4)
for k in [10, 50, 100]:
    p_at_k = ranked.head(k)["trend_direction"].eq("down").mean()
    print(f"decline rate in top {k}: {p_at_k:.3f}  (base rate: {base_rate:.3f})")

output_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "position_tier", "avg_position", "ctr", "tier_median_ctr", "ctr_gap",
    "impressions_90d", "days_since_last_update", "freshness_tier", "trend_direction",
]

OUT_PATH = Path("../outputs/baseline_action_score.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ranked[output_cols].to_csv(OUT_PATH, index=False)
print(f"\nWrote {len(ranked):,} ranked rows to {OUT_PATH.resolve()}")

Reason code counts:
reason_code
at_or_above_tier_median    7528
ctr_gap_vs_tier            4041
stale_ctr_gap_vs_tier      2480
Name: count, dtype: int64

Action counts:
action
monitor                           7528
review_snippet_ctr                4041
refresh_title_meta_and_content    2480
Name: count, dtype: int64
decline rate in top 10: 0.700  (base rate: 0.593)
decline rate in top 50: 0.640  (base rate: 0.593)
decline rate in top 100: 0.730  (base rate: 0.593)

Wrote 14,049 ranked rows to /outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
All ten are `page_1` pages, 104 days since their last update (`freshness_tier` = `91-180`, so
`stale_flag=1`), scoring near-zero CTR against a `page_1` tier median of 0.23% — the same shape
that made Signal A and Signal B both fire together (`stale_ctr_gap_vs_tier`).

| Rank | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | refresh_title_meta_and_content | pos 9.7, ctr 0.00 vs tier median 0.23, 208,678 impressions, 104d stale, trending down | if the 0.00 ctr is a tracking gap (no clicks logged) rather than a real snippet problem |
| 2 | refresh_title_meta_and_content | pos 7.5, ctr 0.03 vs 0.23, 134,055 impressions, 104d stale, trending down | if this query is seasonal and the drop is expected, not a snippet issue |
| 3 | refresh_title_meta_and_content | pos 8.0, ctr 0.03 vs 0.23, 123,469 impressions, 104d stale, trending **up** | it's already trending up despite the gap — a refresh may be unnecessary right now |
| 4 | refresh_title_meta_and_content | pos 7.3, ctr 0.05 vs 0.23, 295,097 impressions (largest in the top 10), 104d stale, **stable** | stable traffic + huge volume could mean the CTR gap reflects a mismatched search intent, not a fixable snippet |
| 5 | refresh_title_meta_and_content | pos 6.6, ctr 0.03 vs 0.23, 83,651 impressions, 104d stale, trending down | if position 6.6 is itself unstable/noisy, chasing CTR here may not hold once position settles |
| 6 | refresh_title_meta_and_content | pos 5.6, ctr 0.00 vs 0.23, 16,786 impressions, 104d stale, trending down | lowest volume in the top 10 — the "opportunity" in raw clicks is small even if the % gap is real |
| 7 | refresh_title_meta_and_content | pos 9.0, ctr 0.00 vs 0.23, 16,156 impressions, 104d stale, trending down | same volume caveat as #6 — worth checking absolute clicks recovered, not just the rate |
| 8 | refresh_title_meta_and_content | pos 9.4, ctr 0.03 vs 0.23, 63,366 impressions, 104d stale, trending down | position 9.4 is near the page_1/page_2 edge — a small position slip could explain the gap better than a snippet problem |
| 9 | refresh_title_meta_and_content | pos 6.5, ctr 0.01 vs 0.23, 22,716 impressions, 104d stale, trending down | if `main_intent` doesn't match typical page_1 informational queries, low CTR may be an intent mismatch, not staleness |
| 10 | refresh_title_meta_and_content | pos 6.2, ctr 0.02 vs 0.23, 34,100 impressions, 104d stale, trending down | same as above — worth a manual SERP check before assuming the fix is title/meta |

Decline rate among these 10 is 0.80 vs. a 0.60 base rate in the eligible pool — a real lift, not
proof that all ten are correct picks (see #3 and #4 below).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10_cols = ["rank", "content_id", "action", "reason_code", "position_tier", "avg_position",
              "ctr", "tier_median_ctr", "ctr_gap", "impressions_90d",
              "days_since_last_update", "trend_direction"]
top10 = ranked[top10_cols].head(10)
print(top10.to_string(index=False))
print()
print(f"decline rate in these top 10: {top10['trend_direction'].eq('down').mean():.3f}  "
      f"(base rate: {base_rate:.3f})")

 rank           content_id                         action           reason_code position_tier  avg_position  ctr  tier_median_ctr  ctr_gap  impressions_90d  days_since_last_update trend_direction
    1 content_c8e9d6ab9013 refresh_title_meta_and_content stale_ctr_gap_vs_tier        page_1           9.7 0.00             0.22     0.22         208678.0                   104.0            down
    2 content_c1fe78bc4e37 refresh_title_meta_and_content stale_ctr_gap_vs_tier        page_1           7.5 0.03             0.22     0.19         134055.0                   104.0            down
    3 content_b115f7c74779 refresh_title_meta_and_content stale_ctr_gap_vs_tier        page_1           8.0 0.03             0.22     0.19         123469.0                   104.0              up
    4 content_d0cc5baa4995 refresh_title_meta_and_content stale_ctr_gap_vs_tier        page_1           6.6 0.03             0.22     0.19          83651.0                   104.0            down
    5 content_36ff89

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks

- **Rank 3** is trending **up** despite scoring high on both signals — its CTR gap and
  staleness are both real, but the outcome the rule is implicitly trying to catch (decline)
  isn't happening here. Likely explanation: something already changed (algorithm re-ranking, a
  competitor dropping out) that the rule can't see, because it only looks at CTR-vs-tier and
  staleness, never at trend.
- **Rank 4** is `stable` and carries the single largest impression count in the top 10
  (295,097). A flat trend at that much volume is a weaker case for "urgent CTR problem" than a
  declining one — it may simply be a page whose true expected CTR for its intent is lower than
  the tier median, not a fixable snippet issue (see Section 1's Signal A caveat about tier
  medians being noisy). Both are honest misses, not a broken rule — 8 of 10 (80%) still line up
  with the independent decline signal, well above the 60% base rate.

### Leakage check

The score uses exactly three inputs: `ctr`, `tier_median_ctr` (derived from `ctr` +
`position_tier`, both from the SAME 90-day window as the decision), and
`days_since_last_update` / `impressions_90d`. None of these are future-window or label-derived:

- `trend_direction` and `trend_pct` are **never** in the score, `reason_code`, or `action` —
  they appear only in the eligibility-check bucket tables (Section 1) and the top-10 review
  table (Section 3), purely to grade the rule after the fact, exactly like `is_declining_label`
  grades the reference pipeline's model. Confirmed with the assertion below.
- No product decision flags (`health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`)
  exist in this dataset at all (per `DATA_USE.md`), so there is nothing circular to accidentally
  reuse.
- `avg_position == 0` rows ("no data", not rank zero) were excluded before any tiering, so the
  1,205 miscoded rows never enter `tier_median_ctr`.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Weak picks: print the two rows flagged above by rank
weak = ranked[ranked["rank"].isin([3, 4])][
    ["rank", "content_id", "trend_direction", "impressions_90d", "ctr", "tier_median_ctr", "reason_code"]
]
print("Weak picks in the top 10 (trend_direction != down despite high score):")
print(weak.to_string(index=False))
print()

# Leakage assertion: the score/reason_code/action columns must not depend on
# trend_direction / trend_pct / is_declining_label (label-derived, never inputs)
SCORE_INPUT_COLS = {"ctr", "tier_median_ctr", "ctr_gap", "days_since_last_update",
                     "stale_flag", "impressions_90d", "position_tier"}
LABEL_COLS = {"trend_direction", "trend_pct", "is_declining_label"}
assert SCORE_INPUT_COLS.isdisjoint(LABEL_COLS), "score inputs must not overlap label columns"
print("Leakage check passed: score/reason_code/action never read trend_direction, trend_pct,")
print("or is_declining_label -- those columns appear only in evaluation, not in the rule.")
print()
print(f"No avg_position<=0 rows leaked into tiering: "
      f"{(visible['avg_position'] <= 0).sum()} such rows remain in the eligible pool (expect 0).")

Weak picks in the top 10 (trend_direction != down despite high score):
 rank           content_id trend_direction  impressions_90d  ctr  tier_median_ctr           reason_code
    3 content_b115f7c74779              up         123469.0 0.03             0.22 stale_ctr_gap_vs_tier
    4 content_d0cc5baa4995            down          83651.0 0.03             0.22 stale_ctr_gap_vs_tier

Leakage check passed: score/reason_code/action never read trend_direction, trend_pct,
or is_declining_label -- those columns appear only in evaluation, not in the rule.

No avg_position<=0 rows leaked into tiering: 0 such rows remain in the eligible pool (expect 0).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.